# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## Konfiguracja

Przed uruchomieniem tego notatnika upewnij się, że masz:

1. **Projekt Microsoft Foundry** z wdrożonym modelem czatu (np. `gpt-5-mini`).
2. **Zalogowano się za pomocą Azure CLI** — wpisz `az login` w swoim terminalu.
3. **Ustawione wymagane zmienne środowiskowe:**
   - `AZURE_AI_PROJECT_ENDPOINT` — punkt końcowy Twojego projektu Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nazwa Twojego wdrożonego modelu.

Poniższa komórka instaluje potrzebne pakiety Pythona.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tworzenie Twojego Pierwszego Agenta

Agent potrzebuje dwóch rzeczy:

- **Instrukcji**, które mówią mu *kim jest* i *jak się zachowywać* (systemowy prompt).
- **Narzędzi** — funkcji Pythona oznaczonych dekoratorem `@tool`, które agent może wywoływać, aby uzyskać informacje lub wykonać działania.

Poniżej definiujemy proste narzędzie, które zwraca listę popularnych miejsc na wakacje. Agent będzie korzystał z tego narzędzia, gdy użytkownik poprosi o rekomendacje podróży.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Strumieniowe odpowiedzi

Dla bardziej interaktywnego doświadczenia możesz **strumieniowo** odbierać odpowiedź agenta. Zamiast czekać na pełną odpowiedź, agent dostarcza fragmenty tekstu w miarę ich generowania. Jest to szczególnie przydatne w interfejsach czatu, gdzie chcesz wyświetlać wynik w czasie rzeczywistym.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

- **Utworzyć dostawcę**, który łączy się z Microsoft Foundry Agent Service za pomocą `FoundryChatClient`.
- **Zdefiniować narzędzie** korzystając z dekoratora `@tool`, aby agent mógł wywoływać twoje funkcje Pythona.
- **Uruchomić agenta** z wiadomością od użytkownika i wydrukować jego odpowiedź.
- **Przesyłać odpowiedzi na bieżąco** dla wyjścia w czasie rzeczywistym.

W następnej lekcji zgłębimy bardziej frameworki agentów i nauczymy się, jak zapewnić agentom potężniejsze narzędzia i możliwości wieloetapowego rozumowania.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
